In [ ]:
import datetime
import time
from Parameters import *
mix_ratio = 1
surf_npoint_source = 8192
target_npoint = 2048
args = parser.parse_known_args()[0]
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_index
sys.path.insert(1, os.path.dirname(os.path.abspath(__name__)))
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
import torch
from modelPAR_volumeAL import PointAR
from dataset.FFDshape_volume import FFDshape_ptp_ss
from Loss import LabelSmoothingCE, reg_loss
from Transforms3 import PCDPretreatment, get_data_augment
from Trainer_volume_post import Trainer
from utils import IdentityScheduler

post_dir = r'E:\MHATT_simp7_SS'
stl_base =  'ShapeSource'
shape_dict = {'cap': 'capsule', 'sld': 'slender', 'spl': 'spaceplane', 'x33':'X-33', 'wr': 'waverider'}
cgns_dict = {'cap': 'capsule', 'sld': 'slender', 'spl': 'spaceplane', 'x33':'x-33', 'wr': 'waverider44'}
shape_name = 'sld' # cap, sld, spl, x33, wr, rocket

aoa = 0
aoa_str = '{:.4f}'.format(aoa)
cgns_file = f'{post_dir}\euler_cases\{cgns_dict[shape_name]}\Mach8.0000_Alpha{aoa_str}_Beta0.0000\check\FlowResult_tecplot.cgns'
if not os.path.exists(cgns_file):
    cgns_file.replace('.cgns','.dat')
print(f'read file: {cgns_file}')
pth_file = r'result_train\PAR_h8_model=basic_c_ds=SS2000_aug=basic_lr=0.001_wd=1e-08_bs=16_Adam_cosine\PAR_h8_SS2000_epoch300.pth'
cmd = f'python main_PAR_vol_al.py --name PAR_h8 -bs 16 --lr 3e-4 -ec 5 --dataset none -ne 300 --att_cfg 512 256 128 -en 1 -ts 1400 --optimizer Adam --auto_cast False -cp {pth_file}'
cmd_list = cmd.split()
param_list = []
param_translate = {'-bs':'--batch_size', '-ec':'--eval_cycle', '-ne':'--num_epochs',
                    '-en':'--expand_num', '-ts':'--train_sample_idx','-cp':'--checkpoint'}
# 遍历list
for i in range(len(cmd_list)):
    if cmd_list[i].startswith('-') or cmd_list[i].startswith('--'):
        if cmd_list[i].startswith('-') and not cmd_list[i].startswith('--'):
            key = param_translate[cmd_list[i]]
        else:
            key = cmd_list[i]
        j = 1
        param_list.append(key)
        while (i+j)<len(cmd_list) and not cmd_list[i+j].startswith('-'):
            param_list.append(cmd_list[i+j])
            j = j+1

print(param_list)
timestamp = os.path.getmtime(param_list[-1])
dt = datetime.datetime.fromtimestamp(timestamp)
print(f"pth生成时间：{dt.year}-{dt.month:02d}-{dt.day:02d} {dt.hour:02d}:{dt.minute:02d}:{dt.second:02d}")
args = parser.parse_args(param_list)

def eval_init():
    # 解析参数
    if args.use_cuda and torch.cuda.is_available():
        args.device = torch.device('cuda')
        gpus = list(range(torch.cuda.device_count()))
        torch.cuda.set_device('cuda:{}'.format(gpus[0]))
    else:
        args.device = torch.device('cpu')
    
    att_cfg = args.att_cfg
    print(att_cfg)
    model_cfg = MODEL_CONFIG[args.model_cfg]
    max_input = model_cfg['max_input']
    normal = model_cfg['normal']
    
    if args.optimizer.lower() == 'adamw':
        Optimizer = torch.optim.AdamW
    elif args.optimizer == 'Adam':
        Optimizer = torch.optim.Adam
    elif args.optimizer == 'SGD':
        Optimizer = torch.optim.SGD
    
    if args.scheduler.lower() == 'identity':
        Scheduler = IdentityScheduler
    else:
        args.scheduler = 'cosine'
        Scheduler = torch.optim.lr_scheduler.CosineAnnealingLR

    # 数据变换、加载数据集
    logger.info('Prepare Data')
    '''数据变换、加载数据集'''
    data_augment, random_sample, random_drop = get_data_augment(DATA_AUG_CONFIG[args.data_aug])
   
    transforms = PCDPretreatment(surf_npoint_source=surf_npoint_source, target_npoint=target_npoint, down_sample='random', normal=normal,
                                 data_augmentation=data_augment, random_drop=random_drop, resampling=random_sample, mix_ratio=mix_ratio)

    '''Prepare dataset'''
    if args.dataset_path is None or args.dataset_path == 'default':
        if args.dataset == 'WR_fl':
            # default_dataset_path_list = [r'F:\WR_fl']
            default_dataset_path_list = [r'../MHATT_simp4/dataroot']
        elif args.dataset == 'SS2000':
            default_dataset_path_list = [r'D:\database\shapesffd3']
        # else:
        #     raise ValueError
        
        # for path in default_dataset_path_list:
        #     if os.path.exists(path):
        #         args.dataset_path = path
        #         break
        # else:  # this is for-else block, indent is not missing
        #     raise FileNotFoundError(f'Dataset path not found.')
        logger.info(f'Load default dataset from {args.dataset_path}')
    dataset = None
    if args.dataset == 'WR_fl':
        dataset = FFDshape_ptp(dataroot=args.dataset_path, transforms=transforms,
                                npoints_target=target_npoint, eval_npoints=surf_npoint_source, 
                                expand_num=args.expand_num, tsample=args.train_sample_idx)
    elif args.dataset == 'SS2000':
        dataset = FFDshape_ptp_ss(dataroot=args.dataset_path, transforms=transforms,
                                npoints_target=target_npoint, eval_npoints=surf_npoint_source, 
                                expand_num=args.expand_num, tsample=args.train_sample_idx,
                                gettest=args.gettest)

    # 模型与损失函数
    logger.info('Prepare Models...')
    model = PointAR(att_cfg if att_cfg is not None else [512,256]).to(device=args.device)
    # sampler = PointAR([1], sampler=True).to(device=args.device)
    sampler = None
    if args.optimizer == 'SGD':
        optimizer = Optimizer(model.parameters(), lr=args.lr)
    else:
        optimizer = Optimizer(model.parameters(), lr=args.lr, weight_decay=args.wd)
    scheduler = Scheduler(optimizer, T_max=args.num_epochs, eta_min=0)# eta_min=args.lr * 0.001
    if sampler is not None:
        optimizer_s = Optimizer(sampler.parameters(), lr=args.lr, weight_decay=args.wd)
        scheduler_s = Scheduler(optimizer_s, T_max=args.num_epochs, eta_min=0)
    else:
        optimizer_s = None
        scheduler_s = None
    criterion = reg_loss().to(args.device)
    
    


    # 训练器
    logger.info('Trainer launching...')
    trainer = Trainer(
        args=args,
        model=model,
        sampler=None,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        dataset=dataset,
        mode=args.mode,
        optimizer_s=None,
        scheduler_s=None
    )
    # trainer.post()
    # trainer.run()
    # trainer.test()
    return trainer
eval = eval_init()
model = eval.model

import numpy as np
from sklearn import metrics
import trimesh
device = eval.args.device


stl_file = os.path.join(f'{post_dir}\ShapeSource', f'{shape_dict[shape_name]}.stl')
# stl_file = r'E:\WR_fl\pycode\autofluent\shape_set\waverider\shapes_N80_D30_221_seed1_RTY\shape_075.stl'
mesh_data = trimesh.load(stl_file)
points = mesh_data.vertices
faces = mesh_data.faces
normals = mesh_data.face_normals
xyzn_surf = points[faces, :].mean(1)

# normalize
scale = xyzn_surf[:,:3].max(0) - xyzn_surf[:,:3].min(0)
xyzn_surf[:,:3] = xyzn_surf[:,:3] / scale.max()
points[:,:3] = points[:,:3] / scale.max()

# shift
shift = xyzn_surf[:, :3].mean(0)
xyzn_surf[:,:3] = xyzn_surf[:,:3] - shift
points[:,:3] = points[:,:3] - shift
# check normals
normals_cross = (xyzn_surf - xyzn_surf.mean(0)) * normals
normals_cross = sum(normals_cross.sum(-1)>0)/len(normals_cross)
if normals_cross<-0.7:
    normals = -normals
elif normals_cross>0.7:
    normals = normals
else:
    raise ValueError
# normals = -normals
# to tensor
xyzn_surf = torch.from_numpy(np.concatenate((xyzn_surf, normals), axis=1)).unsqueeze(0).permute(0,2,1).float()

In [ ]:
import logging
logging.basicConfig(level=logging.DEBUG)
import os
import tecplot
import numpy as np
import pandas as pd
import sys
from tecplot.constant import *
tecplot.session.connect(port=7600)
def extract_slice(dataset, dim, origin_list):
    if not isinstance(origin_list, list):
        origin_list = [origin_list]

    normal_dict = {
        0: (1, 0, 0),
        1: (0, 1, 0),
        2: (0, 0, 1)
    }
    x_list = []
    y_list = []
    z_list = []
    # gt_list = []
    ft1_list,ft2_list, ft3_list = [],[],[]
    nodemap_list = []
    for i in range(len(origin_list)):
        origin = tuple(x*origin_list[i] for x in normal_dict.get(dim))
        extracted_slice = tecplot.data.extract.extract_slice(
            origin=origin,
            normal=normal_dict.get(dim),
            source=tecplot.constant.SliceSource.VolumeZones,
            dataset=dataset
        )
        
        extracted_slice.name = 'slice_pred'
        
        tecplot.data.operate.execute_equation('{pred}={CoefPressure}',
            zones=[dataset.zone('slice_pred')]
        )
        
        ft1 = extracted_slice.values('CoefPressure').as_numpy_array()
        ft2 = extracted_slice.values('Density').as_numpy_array()
        ft3 = extracted_slice.values('Mach').as_numpy_array()
        x = extracted_slice.values('CoordinateX').as_numpy_array()
        y = extracted_slice.values('CoordinateY').as_numpy_array()
        z = extracted_slice.values('CoordinateZ').as_numpy_array()
        nodemap = np.array(extracted_slice.nodemap[:])
        
        ft1_list.append(ft1)
        ft2_list.append(ft2)
        ft3_list.append(ft3)

        x_list.append(x)
        y_list.append(y)
        z_list.append(z)
        nodemap_list.append(nodemap)
    return x_list, y_list, z_list, ft1_list,ft2_list, ft3_list, nodemap_list

In [ ]:
# x-plane
# ZONE name: FlowZone1, SurfaceZone1
torch.manual_seed(0)
box_lim = {'y':[0, 0.28], 'z':[-0.05, 0.28]} # x33:{'x':[-1.2, 0.53], 'z':[-0.65, 0.7]}
origin_list = [0.2, -0.1, -0.4]
tecplot.new_layout()
frame = tecplot.active_frame()
dataset = tecplot.data.load_cgns(cgns_file)

# mirror operate
mirror_flow_zone = dataset.zone('FlowZone1').copy()
mirror_flow_zone.name = 'FlowZone1_mirror'
mirror_flow_zone.values('CoordinateY')[:] = -mirror_flow_zone.values('CoordinateY')[:]

x_list, y_list, z_list, ft1_list,ft2_list, ft3_list, nodemap_list = extract_slice(dataset, dim=0, origin_list=[(x+shift[0])*scale.max() for x in origin_list])
# x_list, y_list, z_list, gt_list, nodemap_list
x, y, z = np.concatenate(x_list), np.concatenate(y_list), np.concatenate(z_list)
gt = np.concatenate((np.concatenate(ft1_list).reshape(1,-1),np.concatenate(ft2_list).reshape(1,-1),np.concatenate(ft3_list).reshape(1,-1)), axis=0)
node_count = np.array([len(x) for x in x_list])
node_count = np.cumsum(node_count)
node_count = np.insert(node_count, 0, 0)[:-1]
nodemap = np.concatenate([nodemap+node_c for nodemap, node_c in zip(nodemap_list, node_count)])
xyz_volume = torch.concat((torch.tensor(x.reshape(-1, 1)),
                          torch.tensor(y.reshape(-1, 1)),
                          torch.tensor(z.reshape(-1, 1))), 
                            dim=1)
# # norm & shift
xyz_volume = xyz_volume/ scale.max()
xyz_volume = xyz_volume - shift
xyz_volume = torch.concat((xyz_volume, torch.zeros(xyz_volume.shape[0],3)), dim=1).permute(1,0).unsqueeze(0).float()

# select
remove_pts = (xyz_volume[0,2,:]<box_lim['z'][0]) | (xyz_volume[0,2,:]>box_lim['z'][1]) | (xyz_volume[0,1,:]<box_lim['y'][0]) | (xyz_volume[0,1,:]>box_lim['y'][1])
remove_pts_idx = np.where(remove_pts)[0]

device = 'cpu'
model.to(device)
model.eval()
# Pred

start_time = time.time() # 记录开始时间
with torch.no_grad():
    pred = model(xyzn_surf.to(device), xyz_volume.to(device)).squeeze(0)
end_time = time.time() # 记录结束时间
print('运行时间：', end_time - start_time, '秒')

pred = np.array(pred)
diff = pred-gt
# 统计remove点的出现次数
remove_grid = np.zeros(nodemap.shape[0])
for i in range(len(remove_pts_idx)):
    remove_grid = remove_grid + np.count_nonzero(nodemap == remove_pts_idx[i], axis=1)
nodemap_plot = nodemap[np.where(remove_grid==0)[0],:]

# 调整点
xyz_volume_plot = xyz_volume
xyz_volume_plot[0,0,remove_pts] = 0
xyz_volume_plot[0,1,remove_pts] = 0
xyz_volume_plot[0,2,remove_pts] = 0


In [ ]:
import plotly.graph_objects as go
from sklearn.metrics import r2_score
from plotly.subplots import make_subplots
slice_opacity = 1
percent_alpha = 5
slice_color = 'viridis' # viridis turbo dense
ft_show = 0
mse = np.mean(np.square(gt[:, ~remove_pts]  - pred[:, ~remove_pts]))
corrcoef = np.corrcoef(gt[0, ~remove_pts], pred[0, ~remove_pts])[0][1]
R2 = r2_score(gt[:, ~remove_pts], pred[:, ~remove_pts])
print(f'Slices Mse= {mse}, Corr_1= {corrcoef}, R2_3ft= {R2}')

colorbar_title = ['<i>C<sub>p</sub></i>', 'Rho', 'Mach']
fig = make_subplots(rows=1, cols=3,
                    specs=[[{'is_3d': True}, {'is_3d': True}, {'is_3d': True}]],
                    subplot_titles=['Reference', 'Pred', 'Diff'],
                    )
                    
# color_scale = np.array([np.max([gt[~remove_pts].max(), pred[~remove_pts].max()]), np.min([gt[~remove_pts].min(), pred[~remove_pts].min()])])
color_scale = np.array([np.max([np.percentile(gt[ft_show,~remove_pts], 100-percent_alpha), np.percentile(pred[ft_show,~remove_pts], 100-percent_alpha)]), 
                        np.min([np.percentile(gt[ft_show,~remove_pts], percent_alpha), np.percentile(pred[ft_show,~remove_pts], percent_alpha)])])
print(f'color_scale: {color_scale}')
# color_scale[0] = 0.1
fig.add_trace(go.Mesh3d(
        # 8 vertices of a cube
        x=xyz_volume_plot[0,0,:],
        y=xyz_volume_plot[0,1,:],
        z=xyz_volume_plot[0,2,:],
        colorbar_title=colorbar_title[ft_show],
        # Intensity of each vertex, which will be interpolated and color-coded
        intensity = gt[ft_show, :], # gt pred
        opacity=slice_opacity,
        # i, j and k give the vertices of triangles
        i = np.concatenate((nodemap_plot[:,0], nodemap_plot[:,0])),
        j = np.concatenate((nodemap_plot[:,1], nodemap_plot[:,2])),
        k = np.concatenate((nodemap_plot[:,2], nodemap_plot[:,3])),
        colorbar_x=.3, colorscale=slice_color, 
        colorbar=dict(title_text="Cp", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels"),
        cmax=color_scale[0], cmin=color_scale[1]
    ), 1, 1)

# surf
fig.add_trace(go.Mesh3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    # set the z-value as the intensity for the colorbar
    color='gray',
    opacity=1
), 1, 1)



# 2
fig.add_trace(go.Mesh3d(
        # 8 vertices of a cube
        x=xyz_volume_plot[0,0,:],
        y=xyz_volume_plot[0,1,:],
        z=xyz_volume_plot[0,2,:],
        colorbar_title='<i>C<sub>p</sub></i>',
        colorscale=slice_color,
        # Intensity of each vertex, which will be interpolated and color-coded
        intensity = pred[ft_show, :], # gt pred
        opacity=slice_opacity,
        # i, j and k give the vertices of triangles
        i = np.concatenate((nodemap_plot[:,0], nodemap_plot[:,0])),
        j = np.concatenate((nodemap_plot[:,1], nodemap_plot[:,2])),
        k = np.concatenate((nodemap_plot[:,2], nodemap_plot[:,3])),
        showscale=False,
        cmax=color_scale[0], cmin=color_scale[1]
    ), 1, 2)
# surf
fig.add_trace(go.Mesh3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    # set the z-value as the intensity for the colorbar
    color='gray',
    opacity=1
), 1, 2)

# 3 diff
diff_color_scale = np.array([np.percentile(diff[ft_show,~remove_pts], 100-percent_alpha), np.percentile(diff[ft_show,~remove_pts], percent_alpha)])
diff_color_scale = np.array([max(abs(diff_color_scale)), -max(abs(diff_color_scale))])

fig.add_trace(go.Mesh3d(
        # 8 vertices of a cube
        x=xyz_volume_plot[0,0,:],
        y=xyz_volume_plot[0,1,:],
        z=xyz_volume_plot[0,2,:],
        colorbar_title='Δ',# Δ<i>C<sub>p</sub></i>
        # Intensity of each vertex, which will be interpolated and color-coded
        intensity = diff[ft_show, :], # gt pred diff
        opacity=slice_opacity,
        # i, j and k give the vertices of triangles
        i = np.concatenate((nodemap_plot[:,0], nodemap_plot[:,0])),
        j = np.concatenate((nodemap_plot[:,1], nodemap_plot[:,2])),
        k = np.concatenate((nodemap_plot[:,2], nodemap_plot[:,3])),
        showscale=True,
        colorbar_x=1, colorscale='balance', 
        colorbar=dict(title_text="Cp_diff", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels"),
        cmax=diff_color_scale[0], cmin=diff_color_scale[1]
    ), 1, 3)
# surf
fig.add_trace(go.Mesh3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    # set the z-value as the intensity for the colorbar
    color='gray',
    opacity=1
), 1, 3)

# camera
camera = dict(
    eye=dict(x=-.85, y=-1.5, z=0.65)
)
fig.update_layout(scene1_camera=camera, scene2_camera=camera, scene3_camera=camera)
fig.update_layout(scene1={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
},
scene2={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
},
scene3={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
})

fig.update_layout(scene1=dict(aspectmode='data'), scene2=dict(aspectmode='data'), scene3=dict(aspectmode='data'))# scene1_aspectratio=dict(x=1, y=1, z=2)
fig.update_layout(margin=dict(l=15, r=0.1, b=0))# tight layout b=0, t=10

fig.update_layout(width=1200, height=500)
fig.show()


In [ ]:
# y-plane
# ZONE name: FlowZone1, SurfaceZone1
torch.manual_seed(0)
box_lim = {'x':[-0.92, 0.7], 'z':[-0.65, 0.7]} # x33:{'x':[-1.2, 0.53], 'z':[-0.65, 0.7]}
origin_list = [0]
tecplot.new_layout()
frame = tecplot.active_frame()
dataset = tecplot.data.load_cgns(cgns_file)

# mirror operate
mirror_flow_zone = dataset.zone('FlowZone1').copy()
mirror_flow_zone.name = 'FlowZone1_mirror'
mirror_flow_zone.values('CoordinateY')[:] = -mirror_flow_zone.values('CoordinateY')[:]

x_list, y_list, z_list, ft1_list,ft2_list, ft3_list, nodemap_list = extract_slice(dataset, dim=1, origin_list=[(x+shift[1])*scale.max() for x in origin_list])
# x_list, y_list, z_list, gt_list, nodemap_list
x, y, z = np.concatenate(x_list), np.concatenate(y_list), np.concatenate(z_list)
gt = np.concatenate((np.concatenate(ft1_list).reshape(1,-1),np.concatenate(ft2_list).reshape(1,-1),np.concatenate(ft3_list).reshape(1,-1)), axis=0)
node_count = np.array([len(x) for x in x_list])
node_count = np.cumsum(node_count)
node_count = np.insert(node_count, 0, 0)[:-1]
nodemap = np.concatenate([nodemap+node_c for nodemap, node_c in zip(nodemap_list, node_count)])
xyz_volume = torch.concat((torch.tensor(x.reshape(-1, 1)),
                          torch.tensor(y.reshape(-1, 1)),
                          torch.tensor(z.reshape(-1, 1))), 
                            dim=1)
# # norm & shift
xyz_volume = xyz_volume/ scale.max()
xyz_volume = xyz_volume - shift
xyz_volume = torch.concat((xyz_volume, torch.zeros(xyz_volume.shape[0],3)), dim=1).permute(1,0).unsqueeze(0).float()

# select
remove_pts = (xyz_volume[0,0,:]<box_lim['x'][0]) | (xyz_volume[0,0,:]>box_lim['x'][1]) | (xyz_volume[0,2,:]<box_lim['z'][0]) | (xyz_volume[0,2,:]>box_lim['z'][1])
remove_pts_idx = np.where(remove_pts)[0]

device = 'cpu'
model.to(device)
model.eval()
# Pred
start_time = time.time() # 记录开始时间
with torch.no_grad():
    pred = model(xyzn_surf.to(device), xyz_volume.to(device)).squeeze(0)
end_time = time.time() # 记录结束时间
print('运行时间：', end_time - start_time, '秒')

pred = np.array(pred)
diff = pred-gt
# 统计remove点的出现次数
remove_grid = np.zeros(nodemap.shape[0])
for i in range(len(remove_pts_idx)):
    remove_grid = remove_grid + np.count_nonzero(nodemap == remove_pts_idx[i], axis=1)
nodemap_plot = nodemap[np.where(remove_grid==0)[0],:]

# 调整点
xyz_volume_plot = xyz_volume
xyz_volume_plot[0,0,remove_pts] = 0
xyz_volume_plot[0,1,remove_pts] = 0
xyz_volume_plot[0,2,remove_pts] = 0


In [ ]:
import plotly.graph_objects as go
from sklearn.metrics import r2_score
from plotly.subplots import make_subplots
slice_opacity = 1
percent_alpha = 5
slice_color = 'viridis' # viridis turbo dense
ft_show = 0
mse = np.mean(np.square(gt[:, ~remove_pts]  - pred[:, ~remove_pts]))
corrcoef = np.corrcoef(gt[0, ~remove_pts], pred[0, ~remove_pts])[0][1]
R2 = r2_score(gt[:, ~remove_pts], pred[:, ~remove_pts])
print(f'Slices Mse= {mse}, Corr_1= {corrcoef}, R2_3ft= {R2}')

colorbar_title = ['<i>C<sub>p</sub></i>', 'Rho', 'Mach']
fig = make_subplots(rows=1, cols=3,
                    specs=[[{'is_3d': True}, {'is_3d': True}, {'is_3d': True}]],
                    subplot_titles=['Reference', 'Pred', 'Diff'],
                    )
                    
# color_scale = np.array([np.max([gt[~remove_pts].max(), pred[~remove_pts].max()]), np.min([gt[~remove_pts].min(), pred[~remove_pts].min()])])
color_scale = np.array([np.max([np.percentile(gt[ft_show,~remove_pts], 100-percent_alpha), np.percentile(pred[ft_show,~remove_pts], 100-percent_alpha)]), 
                        np.min([np.percentile(gt[ft_show,~remove_pts], percent_alpha), np.percentile(pred[ft_show,~remove_pts], percent_alpha)])])
print(f'color_scale: {color_scale}')
# color_scale[0] = 0.1
fig.add_trace(go.Mesh3d(
        # 8 vertices of a cube
        x=xyz_volume_plot[0,0,:],
        y=xyz_volume_plot[0,1,:],
        z=xyz_volume_plot[0,2,:],
        colorbar_title=colorbar_title[ft_show],
        # Intensity of each vertex, which will be interpolated and color-coded
        intensity = gt[ft_show, :], # gt pred
        opacity=slice_opacity,
        # i, j and k give the vertices of triangles
        i = np.concatenate((nodemap_plot[:,0], nodemap_plot[:,0])),
        j = np.concatenate((nodemap_plot[:,1], nodemap_plot[:,2])),
        k = np.concatenate((nodemap_plot[:,2], nodemap_plot[:,3])),
        colorbar_x=.3, colorscale=slice_color, 
        colorbar=dict(title_text="Cp", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels"),
        cmax=color_scale[0], cmin=color_scale[1]
    ), 1, 1)

# surf
fig.add_trace(go.Mesh3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    # set the z-value as the intensity for the colorbar
    color='gray',
    opacity=1
), 1, 1)



# 2
fig.add_trace(go.Mesh3d(
        # 8 vertices of a cube
        x=xyz_volume_plot[0,0,:],
        y=xyz_volume_plot[0,1,:],
        z=xyz_volume_plot[0,2,:],
        colorbar_title='<i>C<sub>p</sub></i>',
        colorscale=slice_color,
        # Intensity of each vertex, which will be interpolated and color-coded
        intensity = pred[ft_show, :], # gt pred
        opacity=slice_opacity,
        # i, j and k give the vertices of triangles
        i = np.concatenate((nodemap_plot[:,0], nodemap_plot[:,0])),
        j = np.concatenate((nodemap_plot[:,1], nodemap_plot[:,2])),
        k = np.concatenate((nodemap_plot[:,2], nodemap_plot[:,3])),
        showscale=False,
        cmax=color_scale[0], cmin=color_scale[1]
    ), 1, 2)
# surf
fig.add_trace(go.Mesh3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    # set the z-value as the intensity for the colorbar
    color='gray',
    opacity=1
), 1, 2)

# 3 diff
diff_color_scale = np.array([np.percentile(diff[ft_show,~remove_pts], 100-percent_alpha), np.percentile(diff[ft_show,~remove_pts], percent_alpha)])
diff_color_scale = np.array([max(abs(diff_color_scale)), -max(abs(diff_color_scale))])

fig.add_trace(go.Mesh3d(
        # 8 vertices of a cube
        x=xyz_volume_plot[0,0,:],
        y=xyz_volume_plot[0,1,:],
        z=xyz_volume_plot[0,2,:],
        colorbar_title='Δ',# Δ<i>C<sub>p</sub></i>
        # Intensity of each vertex, which will be interpolated and color-coded
        intensity = diff[ft_show, :], # gt pred diff
        opacity=slice_opacity,
        # i, j and k give the vertices of triangles
        i = np.concatenate((nodemap_plot[:,0], nodemap_plot[:,0])),
        j = np.concatenate((nodemap_plot[:,1], nodemap_plot[:,2])),
        k = np.concatenate((nodemap_plot[:,2], nodemap_plot[:,3])),
        showscale=True,
        colorbar_x=1, colorscale='balance', 
        colorbar=dict(title_text="Cp_diff", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels"),
        cmax=diff_color_scale[0], cmin=diff_color_scale[1]
    ), 1, 3)
# surf
fig.add_trace(go.Mesh3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    # set the z-value as the intensity for the colorbar
    color='gray',
    opacity=1
), 1, 3)

# camera
camera = dict(
    eye=dict(x=0, y=-1.6, z=0)
)
fig.update_layout(scene1_camera=camera, scene2_camera=camera, scene3_camera=camera)
fig.update_layout(scene1={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
},
scene2={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
},
scene3={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
})

fig.update_layout(scene1=dict(aspectmode='data'), scene2=dict(aspectmode='data'), scene3=dict(aspectmode='data'))# scene1_aspectratio=dict(x=1, y=1, z=2)
fig.update_layout(margin=dict(l=15, r=0.1, b=0))# tight layout b=0, t=10

fig.update_layout(width=1200, height=500)
fig.show()


In [ ]:
# # z-plane
# # ZONE name: FlowZone1, SurfaceZone1
# torch.manual_seed(0)
# box_lim = {'x':[-1.2, 0.53], 'y':[-0.63, 0.63]} # x33:{'x':[-1.2, 0.53], 'z':[-0.65, 0.7]}
# origin_list = [0.04]# x33: -0.05
# tecplot.new_layout()
# frame = tecplot.active_frame()
# dataset = tecplot.data.load_cgns(cgns_file)

# # mirror operate
# mirror_flow_zone = dataset.zone('FlowZone1').copy()
# mirror_flow_zone.name = 'FlowZone1_mirror'
# mirror_flow_zone.values('CoordinateY')[:] = -mirror_flow_zone.values('CoordinateY')[:]

# x_list, y_list, z_list, gt_list, nodemap_list = extract_slice(dataset, dim=2, origin_list=[(x+shift[2])*scale.max() for x in origin_list])

# # x_list, y_list, z_list, gt_list, nodemap_list
# x, y, z, gt = np.concatenate(x_list), np.concatenate(y_list), np.concatenate(z_list), np.concatenate(gt_list)
# node_count = np.array([len(x) for x in x_list])
# node_count = np.cumsum(node_count)
# node_count = np.insert(node_count, 0, 0)[:-1]
# nodemap = np.concatenate([nodemap+node_c for nodemap, node_c in zip(nodemap_list, node_count)])
# xyz_volume = torch.concat((torch.tensor(x.reshape(-1, 1)),
#                           torch.tensor(y.reshape(-1, 1)),
#                           torch.tensor(z.reshape(-1, 1))), 
#                             dim=1)
# # # norm & shift
# xyz_volume = xyz_volume/ scale.max()
# xyz_volume = xyz_volume - shift
# xyz_volume = torch.concat((xyz_volume, torch.zeros(xyz_volume.shape[0],3)), dim=1).permute(1,0).unsqueeze(0).float()

# # select
# remove_pts = (xyz_volume[0,0,:]<box_lim['x'][0]) | (xyz_volume[0,0,:]>box_lim['x'][1]) | (xyz_volume[0,1,:]<box_lim['y'][0]) | (xyz_volume[0,1,:]>box_lim['y'][1])
# remove_pts_idx = np.where(remove_pts)[0]

# device = 'cpu'
# model.to(device)
# model.eval()
# # Pred

# start_time = time.time() # 记录开始时间
# with torch.no_grad():
#     pred = model(xyzn_surf.to(device), xyz_volume.to(device)).squeeze(0)
# end_time = time.time() # 记录结束时间
# print('运行时间：', end_time - start_time, '秒')

# pred = np.array(pred.squeeze(0))
# diff = pred-gt

# # 统计remove点的出现次数
# remove_grid = np.zeros(nodemap.shape[0])
# for i in range(len(remove_pts_idx)):
#     remove_grid = remove_grid + np.count_nonzero(nodemap == remove_pts_idx[i], axis=1)
# nodemap_plot = nodemap[np.where(remove_grid==0)[0],:]

# # 调整点
# xyz_volume_plot = xyz_volume
# xyz_volume_plot[0,0,remove_pts] = 0
# xyz_volume_plot[0,1,remove_pts] = 0
# xyz_volume_plot[0,2,remove_pts] = 0


In [ ]:
# import plotly.graph_objects as go
# from sklearn.metrics import r2_score
# from plotly.subplots import make_subplots
# slice_opacity = 1
# percent_alpha = 5
# slice_color = 'turbo' # viridis
# mse = np.mean(np.square(gt[~remove_pts]  - pred[~remove_pts]))
# corrcoef = np.corrcoef(gt[~remove_pts], pred[~remove_pts])[0][1]
# R2 = r2_score(gt[~remove_pts], pred[~remove_pts])
# print(f'Slices Mse= {mse}, Corr= {corrcoef}, R2= {R2}')

# fig = make_subplots(rows=1, cols=3,
#                     specs=[[{'is_3d': True}, {'is_3d': True}, {'is_3d': True}]],
#                     subplot_titles=['Reference', 'Pred', 'Diff'],
#                     )
                    
# # color_scale = np.array([np.max([gt[~remove_pts].max(), pred[~remove_pts].max()]), np.min([gt[~remove_pts].min(), pred[~remove_pts].min()])])
# color_scale = np.array([np.max([np.percentile(gt[~remove_pts], 100-percent_alpha), np.percentile(pred[~remove_pts], 100-percent_alpha)]), 
#                         np.min([np.percentile(gt[~remove_pts], percent_alpha), np.percentile(pred[~remove_pts], percent_alpha)])])
# print(f'color_scale: {color_scale}')
# # color_scale[0] = 0.1
# fig.add_trace(go.Mesh3d(
#         # 8 vertices of a cube
#         x=xyz_volume_plot[0,0,:],
#         y=xyz_volume_plot[0,1,:],
#         z=xyz_volume_plot[0,2,:],
#         colorbar_title='<i>C<sub>p</sub></i>',
#         # Intensity of each vertex, which will be interpolated and color-coded
#         intensity = gt, # gt pred
#         opacity=slice_opacity,
#         # i, j and k give the vertices of triangles
#         i = np.concatenate((nodemap_plot[:,0], nodemap_plot[:,0])),
#         j = np.concatenate((nodemap_plot[:,1], nodemap_plot[:,2])),
#         k = np.concatenate((nodemap_plot[:,2], nodemap_plot[:,3])),
#         colorbar_x=.3, colorscale=slice_color, 
#         colorbar=dict(title_text="Cp", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels"),
#         cmax=color_scale[0], cmin=color_scale[1]
#     ), 1, 1)

# # surf
# fig.add_trace(go.Mesh3d(
#     x=points[:, 0], y=points[:, 1], z=points[:, 2],
#     i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
#     # set the z-value as the intensity for the colorbar
#     color='gray',
#     opacity=1
# ), 1, 1)



# # 2
# fig.add_trace(go.Mesh3d(
#         # 8 vertices of a cube
#         x=xyz_volume_plot[0,0,:],
#         y=xyz_volume_plot[0,1,:],
#         z=xyz_volume_plot[0,2,:],
#         colorbar_title='<i>C<sub>p</sub></i>',
#         colorscale=slice_color,
#         # Intensity of each vertex, which will be interpolated and color-coded
#         intensity = pred, # gt pred
#         opacity=slice_opacity,
#         # i, j and k give the vertices of triangles
#         i = np.concatenate((nodemap_plot[:,0], nodemap_plot[:,0])),
#         j = np.concatenate((nodemap_plot[:,1], nodemap_plot[:,2])),
#         k = np.concatenate((nodemap_plot[:,2], nodemap_plot[:,3])),
#         showscale=False,
#         cmax=color_scale[0], cmin=color_scale[1]
#     ), 1, 2)
# # surf
# fig.add_trace(go.Mesh3d(
#     x=points[:, 0], y=points[:, 1], z=points[:, 2],
#     i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
#     # set the z-value as the intensity for the colorbar
#     color='gray',
#     opacity=1
# ), 1, 2)

# # 3 diff
# diff_color_scale = np.array([np.percentile(diff[~remove_pts], 100-percent_alpha), np.percentile(diff[~remove_pts], percent_alpha)])
# diff_color_scale = np.array([max(abs(diff_color_scale)), -max(abs(diff_color_scale))])

# fig.add_trace(go.Mesh3d(
#         # 8 vertices of a cube
#         x=xyz_volume_plot[0,0,:],
#         y=xyz_volume_plot[0,1,:],
#         z=xyz_volume_plot[0,2,:],
#         colorbar_title='Δ<i>C<sub>p</sub></i>',
#         # Intensity of each vertex, which will be interpolated and color-coded
#         intensity = diff, # gt pred diff
#         opacity=slice_opacity,
#         # i, j and k give the vertices of triangles
#         i = np.concatenate((nodemap_plot[:,0], nodemap_plot[:,0])),
#         j = np.concatenate((nodemap_plot[:,1], nodemap_plot[:,2])),
#         k = np.concatenate((nodemap_plot[:,2], nodemap_plot[:,3])),
#         showscale=True,
#         colorbar_x=1, colorscale='rdbu', 
#         colorbar=dict(title_text="Cp_diff", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels"),
#         cmax=diff_color_scale[0], cmin=diff_color_scale[1]
#     ), 1, 3)
# # surf
# fig.add_trace(go.Mesh3d(
#     x=points[:, 0], y=points[:, 1], z=points[:, 2],
#     i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
#     # set the z-value as the intensity for the colorbar
#     color='gray',
#     opacity=1
# ), 1, 3)

# # camera
# camera = dict(
#     up=dict(x=0, y=1., z=0),
#     eye=dict(x=0, y=0, z=1.6)
# )
# fig.update_layout(scene1_camera=camera, scene2_camera=camera, scene3_camera=camera)
# fig.update_layout(scene1={
#     "camera": {
#         "projection": {
#             "type": "orthographic"
#         }
#     },
#     "xaxis": {"showticklabels": False},
#     "yaxis": {"showticklabels": False},
#     "zaxis": {"showticklabels": False}
# },
# scene2={
#     "camera": {
#         "projection": {
#             "type": "orthographic"
#         }
#     },
#     "xaxis": {"showticklabels": False},
#     "yaxis": {"showticklabels": False},
#     "zaxis": {"showticklabels": False}
# },
# scene3={
#     "camera": {
#         "projection": {
#             "type": "orthographic"
#         }
#     },
#     "xaxis": {"showticklabels": False},
#     "yaxis": {"showticklabels": False},
#     "zaxis": {"showticklabels": False}
# })

# fig.update_layout(scene1=dict(aspectmode='data'), scene2=dict(aspectmode='data'), scene3=dict(aspectmode='data'))# scene1_aspectratio=dict(x=1, y=1, z=2)
# fig.update_layout(margin=dict(l=15, r=0.1, b=0))# tight layout b=0, t=10

# fig.update_layout(width=1200, height=500)
# fig.show()


In [ ]:
# surface
# surface容易出现漏洞状的可视化结果，尤其是在mirror侧，怀疑是mirror缺少拓扑的原因
# 考虑用单侧进行线性插值（vol to surf），再对单侧surf镜像化，再使用镜像化的surf进行插值。
# ZONE name: FlowZone1, SurfaceZone1
torch.manual_seed(0)
tecplot.new_layout()
frame = tecplot.active_frame()
dataset = tecplot.data.load_cgns(cgns_file)

# mirror operate
mirror_flow_zone = dataset.zone('FlowZone1').copy()
mirror_flow_zone.name = 'FlowZone1_mirror'
mirror_flow_zone.values('CoordinateY')[:] = -mirror_flow_zone.values('CoordinateY')[:]
# interpolate
tecplot.data.operate.interpolate_linear(dataset.zone('SurfaceZone1'), 
                                        source_zones=[dataset.zone('FlowZone1'), dataset.zone('FlowZone1_mirror')])
# tecplot.data.operate.interpolate_inverse_distance(dataset.zone('SurfaceZone1'), 
#                                         source_zones=[dataset.zone('FlowZone1'), dataset.zone('FlowZone1_mirror')])

gt = dataset.zone('SurfaceZone1').values('CoefPressure').as_numpy_array()
x = dataset.zone('SurfaceZone1').values('CoordinateX').as_numpy_array()
y = dataset.zone('SurfaceZone1').values('CoordinateY').as_numpy_array()
z = dataset.zone('SurfaceZone1').values('CoordinateZ').as_numpy_array()
nodemap = np.array(dataset.zone('SurfaceZone1').nodemap[:])
xyz_volume = torch.concat((torch.tensor(x.reshape(-1, 1)),
                          torch.tensor(y.reshape(-1, 1)),
                          torch.tensor(z.reshape(-1, 1))), 
                            dim=1)
# # norm & shift
xyz_volume = xyz_volume/ scale.max()
xyz_volume = xyz_volume - shift
xyz_volume = torch.concat((xyz_volume, torch.zeros(xyz_volume.shape[0],3)), dim=1).permute(1,0).unsqueeze(0).float()
device = 'cpu'
model.to(device)
model.eval()
# Pred

start_time = time.time() # 记录开始时间
with torch.no_grad():
    pred = model(xyzn_surf.to(device), xyz_volume.to(device)).squeeze(0)
end_time = time.time() # 记录结束时间
print('运行时间：', end_time - start_time, '秒')

pred = np.array(pred[0,:])
diff = pred-gt
remove_pts = np.zeros(pred.shape[0], dtype=bool)
nodemap_plot = nodemap
xyz_volume_plot = xyz_volume

In [ ]:
import plotly.graph_objects as go
from sklearn.metrics import r2_score
from plotly.subplots import make_subplots
slice_opacity = 1
percent_alpha = 5
slice_color = 'turbo' # viridis
mse = np.mean(np.square(gt[~remove_pts]  - pred[~remove_pts]))
corrcoef = np.corrcoef(gt[~remove_pts], pred[~remove_pts])[0][1]
R2 = r2_score(gt[~remove_pts], pred[~remove_pts])
print(f'Slices Mse= {mse}, Corr= {corrcoef}, R2= {R2}')

fig = make_subplots(rows=1, cols=3,
                    specs=[[{'is_3d': True}, {'is_3d': True}, {'is_3d': True}]],
                    subplot_titles=['Reference', 'Pred', 'Diff'],
                    )
                    
# color_scale = np.array([np.max([gt[~remove_pts].max(), pred[~remove_pts].max()]), np.min([gt[~remove_pts].min(), pred[~remove_pts].min()])])
color_scale = np.array([np.max([np.percentile(gt[~remove_pts], 100-percent_alpha), np.percentile(pred[~remove_pts], 100-percent_alpha)]), 
                        np.min([np.percentile(gt[~remove_pts], percent_alpha), np.percentile(pred[~remove_pts], percent_alpha)])])
print(f'color_scale: {color_scale}')
# color_scale[0] = 0.1
fig.add_trace(go.Mesh3d(
        # 8 vertices of a cube
        x=xyz_volume_plot[0,0,:],
        y=xyz_volume_plot[0,1,:],
        z=xyz_volume_plot[0,2,:],
        colorbar_title='<i>C<sub>p</sub></i>',
        # Intensity of each vertex, which will be interpolated and color-coded
        intensity = gt, # gt pred
        opacity=slice_opacity,
        # i, j and k give the vertices of triangles
        i = nodemap_plot[:,0],
        j = nodemap_plot[:,1],
        k = nodemap_plot[:,2],
        colorbar_x=.3, colorscale=slice_color, 
        colorbar=dict(title_text="Cp", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels"),
        cmax=color_scale[0], cmin=color_scale[1]
    ), 1, 1)



# 2
fig.add_trace(go.Mesh3d(
        # 8 vertices of a cube
        x=xyz_volume_plot[0,0,:],
        y=xyz_volume_plot[0,1,:],
        z=xyz_volume_plot[0,2,:],
        colorbar_title='<i>C<sub>p</sub></i>',
        colorscale=slice_color,
        # Intensity of each vertex, which will be interpolated and color-coded
        intensity = pred, # gt pred
        opacity=slice_opacity,
        # i, j and k give the vertices of triangles
        i = nodemap_plot[:,0],
        j = nodemap_plot[:,1],
        k = nodemap_plot[:,2],
        showscale=False,
        cmax=color_scale[0], cmin=color_scale[1]
    ), 1, 2)

# 3 diff
diff_color_scale = np.array([np.percentile(diff[~remove_pts], 100-percent_alpha), np.percentile(diff[~remove_pts], percent_alpha)])
diff_color_scale = np.array([max(abs(diff_color_scale)), -max(abs(diff_color_scale))])

fig.add_trace(go.Mesh3d(
        # 8 vertices of a cube
        x=xyz_volume_plot[0,0,:],
        y=xyz_volume_plot[0,1,:],
        z=xyz_volume_plot[0,2,:],
        colorbar_title='Δ<i>C<sub>p</sub></i>',
        # Intensity of each vertex, which will be interpolated and color-coded
        intensity = diff, # gt pred diff
        opacity=slice_opacity,
        # i, j and k give the vertices of triangles
        i = nodemap_plot[:,0],
        j = nodemap_plot[:,1],
        k = nodemap_plot[:,2],
        showscale=True,
        colorbar_x=1, colorscale='rdbu', 
        colorbar=dict(title_text="Cp_diff", len=200, lenmode="pixels", thickness=10, thicknessmode="pixels"),
        cmax=diff_color_scale[0], cmin=diff_color_scale[1]
    ), 1, 3)

# camera
camera = dict(
    eye=dict(x=-.85, y=-1.5, z=0.65)
)
fig.update_layout(scene1_camera=camera, scene2_camera=camera, scene3_camera=camera)
fig.update_layout(scene1={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
},
scene2={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
},
scene3={
    "camera": {
        "projection": {
            "type": "orthographic"
        }
    },
    "xaxis": {"showticklabels": False},
    "yaxis": {"showticklabels": False},
    "zaxis": {"showticklabels": False}
})

fig.update_layout(scene1=dict(aspectmode='data'), scene2=dict(aspectmode='data'), scene3=dict(aspectmode='data'))# scene1_aspectratio=dict(x=1, y=1, z=2)
fig.update_layout(margin=dict(l=15, r=0.1, b=0))# tight layout b=0, t=10

fig.update_layout(width=1200, height=500)
fig.show()


In [ ]:
# # surface
# # ZONE name: FlowZone1, SurfaceZone1
# torch.manual_seed(0)
# tecplot.new_layout()
# frame = tecplot.active_frame()
# dataset = tecplot.data.load_cgns(cgns_file)
# current_dataset = tecplot.active_frame().dataset
# tecplot.data.operate.execute_equation('{pred}={CoordinateX}',
#     zones=[current_dataset.zone('FlowZone1')])
# x = dataset.zone('FlowZone1').values('CoordinateX').as_numpy_array()
# y = dataset.zone('FlowZone1').values('CoordinateY').as_numpy_array()
# z = dataset.zone('FlowZone1').values('CoordinateZ').as_numpy_array()

# xyz_volume = torch.concat((torch.tensor(x.reshape(-1, 1)),
#                           torch.tensor(y.reshape(-1, 1)),
#                           torch.tensor(z.reshape(-1, 1))), 
#                             dim=1)
# # # norm & shift
# xyz_volume = xyz_volume/ scale.max()
# xyz_volume = xyz_volume - shift
# xyz_volume = torch.concat((xyz_volume, torch.zeros(xyz_volume.shape[0],3)), dim=1).permute(1,0).unsqueeze(0).float()
# device = 'cpu'
# model.to(device)
# model.eval()
# pred = torch.zeros((1,xyz_volume.shape[-1]))



In [ ]:
# start_pt = 0
# end_pt = 0
# vol_len_lim = 150000
# start_time = time.time() # 记录开始时间
# while end_pt!=xyz_volume.shape[-1]:
#     end_pt = min(xyz_volume.shape[-1], start_pt+vol_len_lim)
#     print(end_pt/xyz_volume.shape[-1])
#     with torch.no_grad():
#         pred[:1, start_pt:end_pt] = model(xyzn_surf.to(device), xyz_volume[:,:,start_pt:end_pt].to(device)).squeeze(0)
#     start_pt = end_pt
# end_time = time.time() # 记录结束时间
# pred = np.array(pred.squeeze(0))

In [ ]:
# dataset.zone('FlowZone1').values('pred')[:] = pred
# mirror_flow_zone = dataset.zone('FlowZone1').copy()
# mirror_flow_zone.name = 'FlowZone1_mirror'
# mirror_flow_zone.values('CoordinateY')[:] = -mirror_flow_zone.values('CoordinateY')[:]
# # interpolate
# tecplot.data.operate.interpolate_linear(dataset.zone('SurfaceZone1'), 
#                                         source_zones=[dataset.zone('FlowZone1'), dataset.zone('FlowZone1_mirror')])

In [ ]:




# # mirror operate
# mirror_flow_zone = dataset.zone('FlowZone1').copy()
# mirror_flow_zone.name = 'FlowZone1_mirror'
# mirror_flow_zone.values('CoordinateY')[:] = -mirror_flow_zone.values('CoordinateY')[:]
# # interpolate
# tecplot.data.operate.interpolate_linear(dataset.zone('SurfaceZone1'), 
#                                         source_zones=[dataset.zone('FlowZone1'), dataset.zone('FlowZone1_mirror')])
# # tecplot.data.operate.interpolate_inverse_distance(dataset.zone('SurfaceZone1'), 
# #                                         source_zones=[dataset.zone('FlowZone1'), dataset.zone('FlowZone1_mirror')])

# gt = dataset.zone('SurfaceZone1').values('CoefPressure').as_numpy_array()
# x = dataset.zone('SurfaceZone1').values('CoordinateX').as_numpy_array()
# y = dataset.zone('SurfaceZone1').values('CoordinateY').as_numpy_array()
# z = dataset.zone('SurfaceZone1').values('CoordinateZ').as_numpy_array()
# nodemap = np.array(dataset.zone('SurfaceZone1').nodemap[:])
# xyz_volume = torch.concat((torch.tensor(x.reshape(-1, 1)),
#                           torch.tensor(y.reshape(-1, 1)),
#                           torch.tensor(z.reshape(-1, 1))), 
#                             dim=1)
# # # norm & shift
# xyz_volume = xyz_volume/ scale.max()
# xyz_volume = xyz_volume - shift
# xyz_volume = torch.concat((xyz_volume, torch.zeros(xyz_volume.shape[0],3)), dim=1).permute(1,0).unsqueeze(0).float()
# device = 'cpu'
# model.to(device)
# model.eval()
# # Pred

# start_time = time.time() # 记录开始时间
# with torch.no_grad():
#     pred = model(xyzn_surf.to(device), xyz_volume.to(device)).squeeze(0)
# end_time = time.time() # 记录结束时间
# print('运行时间：', end_time - start_time, '秒')

# pred = np.array(pred.squeeze(0))
# diff = pred-gt
# remove_pts = np.zeros(pred.shape[0], dtype=bool)
# nodemap_plot = nodemap
# xyz_volume_plot = xyz_volume